In [1]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchmetrics.classification import AUROC
from torchvision.datasets import Imagenette, SVHN
from torchvision import models

import logging

import importlib
import FL_sim
import models_to_train

importlib.reload(FL_sim)
importlib.reload(models_to_train)
from models_to_train import ResNetPLModel
from FL_sim import FLSimulator


torch.set_float32_matmul_precision('high')

In [2]:
# dataset = [
#     Imagenette(
#         root='./data/Imagenet', split=s,
#         transform=transforms.Compose([
#             transforms.Resize(256),
#             transforms.CenterCrop(224),
#             transforms.ToTensor(),
#             transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
#         ])
#     ) for s in ['train', 'val']]

dataset = [
    SVHN(
        root='./data/SVHN', split=s,
        transform=transforms.Compose([
            transforms.Resize(32),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    ) for s in ['train', 'test']]

In [3]:
def test_model_training():
    from pytorch_lightning import Trainer
    model = ResNetPLModel(num_classes=10,
        resnet_version='resnet18', lr=0.005)
    training_dataloader = torch.utils.data.DataLoader(
        dataset[0], batch_size=14000, shuffle=True,
        num_workers=10, persistent_workers=True)
    test_dataloader = torch.utils.data.DataLoader(
        dataset[1], batch_size=14000,
        num_workers=2, persistent_workers=True)
    trainer = Trainer(max_epochs=10, accelerator='cuda', log_every_n_steps=len(training_dataloader)//2)
    trainer.fit(model, training_dataloader, test_dataloader)
# test_model_training()

In [4]:
# dataset = [torch.utils.data.Subset(d, list(range(200))) for d in dataset]

In [5]:
def pre_send_process(single_model_grad_agent):
    return single_model_grad_agent


def server_rec_process(all_encoded_model_grad):
    return all_encoded_model_grad

In [6]:
k = 2  # Number of workers
bs = 14000
# bs /= 50 # imagenet
# bs /= 3 # resnet 50
bs = int(bs)

sim = FLSimulator(
    num_agents=k, communication_rounds=3, client_epochs_per_round=30,
    batch_size=bs, dataset_train=dataset[0], dataset_test=dataset[1],
    pl_model=ResNetPLModel(num_classes=10, resnet_version='resnet18',
                           logging_disabled=True, record_gradients=True),
    aggregation_method='fedavg', iid_data=True,
    pre_send_process=pre_send_process,
    server_rec_process=server_rec_process,
)

In [7]:
logging.getLogger("pytorch_lightning").setLevel(logging.WARNING)

sim.run_simulation()


round 1/3 --------------------
  - reporting global model metrics
         test loss: 2.349, test auc: 0.502
         train loss: 2.343, train auc: 0.507
     > training agent 1/2


D:\User\Software\MiniConda3\envs\SuperCondaEnv\Lib\site-packages\pytorch_lightning\trainer\configuration_validator.py:74: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.


RuntimeError: [enforce fail at inline_container.cc:595] . unexpected pos 64 vs 0